In [5]:
INPUT_FILE = './data/Svodniy_otchet_po_chasam_intensivnost+skorost_01.01.2022_08.04.2025_1744117742.csv'

In [6]:
import csv
import os
import re
from dateutil.parser import parse

# === НАСТРОЙКИ ===
OUTPUT_BASE = "output_data/"
os.makedirs(OUTPUT_BASE, exist_ok=True)

# === РЕГУЛЯРКИ ===
km_re = re.compile(r'км \d+\+\d+', re.IGNORECASE)

# === ФУНКЦИИ ===


def safe_filename(s: str) -> str:
    s = s.strip()
    s = s.replace('"', '')
    s = s.replace(' ', '_')
    s = s.replace('/', '_')
    s = re.sub(r'[\\:*?"<>|]', '', s)
    s = re.sub(r'_+', '_', s)
    return s


def is_datetime(string) -> bool:
    try:
        parse(string, fuzzy=False)
        return True
    except (ValueError, TypeError):
        return False


def is_valid_highway_line(s: str) -> bool:
    s = s.strip()
    if not s or s.lower() in ["итого", "среднее", "%"]:
        return False
    if km_re.search(s):
        return False
    if is_datetime(s):
        return False
    return True

# === ОСНОВНОЙ ЦИКЛ ===


current_highway = None
current_km = None
open_file = None
csv_writer = None

with open(INPUT_FILE, "r", encoding="utf-8-sig", newline="") as infile:
    reader = list(csv.reader(infile))

    for i, row in enumerate(reader):
        if not row or not row[0]:
            continue

        first_cell = row[0].strip()

        # Обнаружение новой КМО
        if km_re.search(first_cell):
            current_km = safe_filename(first_cell)

            # Попытка взять трассу из предыдущей строки
            if i > 0:
                prev_row = reader[i - 1]
                prev_val = prev_row[0].strip(
                ) if prev_row and prev_row[0] else ""
                if is_valid_highway_line(prev_val):
                    current_highway = safe_filename(prev_val)
                # иначе — используем last known good current_highway

            # Пропустить, если трасса не определена
            if not current_highway:
                print(
                    f"[!] Пропущен блок КМО: {current_km} (трасса не определена)")
                continue

            # Закрываем предыдущий файл
            if open_file:
                open_file.close()

            # Открываем новый файл
            highway_dir = os.path.join(OUTPUT_BASE, current_highway)
            os.makedirs(highway_dir, exist_ok=True)
            file_path = os.path.join(highway_dir, f"{current_km}.csv")
            open_file = open(file_path, "w", newline="", encoding="utf-8-sig")
            csv_writer = csv.writer(open_file)

            continue

        # Обработка строки с данными
        if is_datetime(first_cell) and csv_writer:
            csv_writer.writerow(row)

# Закрыть последний файл
if open_file:
    open_file.close()

In [9]:
import os

OUTPUT_BASE = "output_data/"

highway_km_dict = {}

for root, dirs, files in os.walk(OUTPUT_BASE):
    for dir_name in dirs:
        dir_path = os.path.join(root, dir_name)
        km_files = []

        for file in os.listdir(dir_path):
            if file.endswith(".csv"):
                km_name = os.path.splitext(file)[0]  # убрать .csv
                km_files.append(km_name)

        highway_km_dict[dir_name] = sorted(km_files)

# Пример вывода
for highway, kms in highway_km_dict.items():
    print(f"{highway}: {len(kms)} км-точек")
    for km in kms:
        print(f"  - {km}")

А-289_Краснодар_-_Славянск-на-Кубани_-_Темрюк: 3 км-точек
  - км_37+338_а_д_А-289_Краснодар_-_Темрюк
  - км_69+370_а_д_А-289_Краснодар_-_Темрюк
  - км_7+350_а_д_А-289_Краснодар_-_Темрюк
А-146_Краснодар_-_Новороссийск_-_Верхнебаканский: 3 км-точек
  - км_130+810_а_д_А-146_Краснодар-Верхнебаканский
  - км_32+000_а_д_А-146_Краснодар-Верхнебаканский
  - км_53+800_а_д_А-146_Краснодар-Верхнебаканский
А-160_Майкоп_-_Усть-Лабинск_-_Кореновск: 1 км-точек
  - км_58+990_а_д_А-160_Майкоп-Кореновск
А-290_Новороссийск_-_Керченский_пролив_-_Керчь: 2 км-точек
  - км_10+200_а_д_А-290_Новороссийск-Керчь
  - км_28+500_а_д_А-290_Новороссийск-Керчь
03_ОП_РЗ_03К-001_г.Краснодар_-_г.Ейск: 1 км-точек
  - км_47+243_а_д_г.Краснодар_-_г.Ейск
М-4_Дон_Москва_–_Воронеж_–_Ростов-на-Дону_–_Краснодар_–_Новороссийск: 5 км-точек
  - км_1278+575_а_д_М-4_Дон_Москва_–_Новороссийск
  - км_1301+500_а_д_М-4_Дон_Москва_–_Новороссийск
  - км_1389+000_а_д_М-4_Дон_Москва_–_Новороссийск
  - км_1430+150_а_д_М-4_Дон_Москва_–_Новорос

In [8]:
highway_km_dict

{'А-289_Краснодар_-_Славянск-на-Кубани_-_Темрюк': ['км_37+338_а_д_А-289_Краснодар_-_Темрюк',
  'км_69+370_а_д_А-289_Краснодар_-_Темрюк',
  'км_7+350_а_д_А-289_Краснодар_-_Темрюк'],
 'А-146_Краснодар_-_Новороссийск_-_Верхнебаканский': ['км_130+810_а_д_А-146_Краснодар-Верхнебаканский',
  'км_32+000_а_д_А-146_Краснодар-Верхнебаканский',
  'км_53+800_а_д_А-146_Краснодар-Верхнебаканский'],
 'А-160_Майкоп_-_Усть-Лабинск_-_Кореновск': ['км_58+990_а_д_А-160_Майкоп-Кореновск'],
 'А-290_Новороссийск_-_Керченский_пролив_-_Керчь': ['км_10+200_а_д_А-290_Новороссийск-Керчь',
  'км_28+500_а_д_А-290_Новороссийск-Керчь'],
 '03_ОП_РЗ_03К-001_г.Краснодар_-_г.Ейск': ['км_47+243_а_д_г.Краснодар_-_г.Ейск'],
 'М-4_Дон_Москва_–_Воронеж_–_Ростов-на-Дону_–_Краснодар_–_Новороссийск': ['км_1278+575_а_д_М-4_Дон_Москва_–_Новороссийск',
  'км_1301+500_а_д_М-4_Дон_Москва_–_Новороссийск',
  'км_1389+000_а_д_М-4_Дон_Москва_–_Новороссийск',
  'км_1430+150_а_д_М-4_Дон_Москва_–_Новороссийск',
  'км_1443+500_а_д_М-4_Дон_Мо